In [ ]:
# import all required libraries
import sys, os
import numpy as np
import pandas as pd
import random
from random import shuffle, choice
import time
import os
import glob
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers
from random import shuffle, choice
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# define a function to build a CNN for the SNP data.
def create_cnn(xtest, regularizer=None):
  # obtain the input dimensions.
  inputShape = (xtest.shape[1], xtest.shape[2])
  inputs = Input(shape=inputShape)
  x = inputs
  # first convolutional layer, remember to remove bias if you are intercalating with batch normalization.
  x = Conv1D(256, kernel_size=3, activation='relu', use_bias=False)(x)
  # batch normalization.
  x = BatchNormalization()(x)
  # second layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # third layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # pool the CNN outputs.
  x = GlobalMaxPooling1D()(x)
  # this part is similar to the MLP, a fully connected neural network. We intercalated with dropout to reduce overfitting.
  x = Dense(128, activation='relu')(x)
  # dropout.
  x = Dropout(0.5)(x)
  # second layer of the fully connected neural network.
  x = Dense(128, activation='relu')(x)
  x = Dropout(0.5)(x)
  # third layer of the fully connected neural network. This one matches the number of nodes coming out of the MLP.
  x = Dense(64, activation='relu')(x)
  # Construct the CNN
  #x = BatchNormalization()(x)#Not working very well
  #x = LayerNormalization()(x)#Better?
  model = Model(inputs, x)
  # Return the CNN
  return model

class GatedConcatenate(Layer):
    """
    Applies a trainable or fixed gate (weight) to each input branch
    before concatenating them.
    
    Args:
        initial_traits_weight (float): The starting weight for the first input (traits).
                                     Must be between 0 and 1. The weight for the
                                     second input (SNPs) will be (1 - this value).
        trainable_gates (bool): If True, the model can learn to adjust these
                                weights. If False, the weights are fixed.
    """
    def __init__(self, initial_traits_weight, trainable_gates=True, **kwargs):
        super(GatedConcatenate, self).__init__(**kwargs)
        if not (0 <= initial_traits_weight <= 1):
            raise ValueError("initial_traits_weight must be between 0 and 1.")
            
        self.initial_weights = [initial_traits_weight, 1.0 - initial_traits_weight]
        self.trainable_gates = trainable_gates

    def build(self, input_shape):
        # Create the gate variables. They are shaped for broadcasting across the features.
        self.gates = self.add_weight(
            name='gates',
            shape=(1, len(input_shape)), # Shape will be (1, 2)
            initializer=tf.constant_initializer(self.initial_weights),
            trainable=self.trainable_gates,
            dtype=tf.float32
        )
        super(GatedConcatenate, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("GatedConcatenate expects a list of exactly two input tensors.")
        
        # Apply the gates (weights) to each branch using element-wise multiplication
        gated_traits = inputs[0] * self.gates[0, 0]
        gated_snps = inputs[1] * self.gates[0, 1]
        
        # Concatenate the scaled branches
        return Concatenate()([gated_traits, gated_snps])

    def get_config(self):
        # Needed for saving/loading the model
        config = super().get_config()
        config.update({
            'initial_traits_weight': self.initial_weights[0],
            'trainable_gates': self.trainable_gates,
        })
        return config
    
def gated_contributions(model, layer_name=None, labels=("traits", "SNPs")):
    # 1) find the layer
    gated_layer = model.get_layer('gated_concatenate')
    weights = gated_layer.get_weights()[0][0]
    rel_weight = np.sum(np.abs(weights))
    print(f"Final learned weights: Traits={weights[0]/rel_weight:.4f}, SNPs={weights[1]/rel_weight:.4f}")

In [2]:
## define variables that will be used to train all networks.
# size of the minibatches containing simulations are passed through the network in each epoch.
batch_size = 256
# number of training iterations (epochs) for the SNP only and the combined networks.
epochs = 100
# number of scenarios being classified.
num_classes = 3

In [ ]:
# load the traits simulated under the BM model for the 3 scenarios. 
traits_BM = []
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
Traits_BM = np.array(traits_BM)

# standard scale the continuous (BM) traits
scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

# transform into a NumPy array. 
Traits_BM = np.array(traits_BM)

# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./trainingSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./trainingSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./trainingSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
u=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

# transform SNP major alleles in -1 and minor in 1.
for arr,array in enumerate(u):
    for idx,row in enumerate(array):
        if np.count_nonzero(row) > len(row)/2:
            u[arr][idx][u[arr][idx] == 1] = -1
            u[arr][idx][u[arr][idx] == 0] = 1
        else:
            u[arr][idx][u[arr][idx] == 0] = -1
            
# create a label vector in the same order as the simulations.
y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2sp']))])
y.extend([2 for i in range(len(u3['Model_3sp']))])
y = np.array(y)

# make sure labels, SNP and traits matrices all have the same length.
print (len(y), len(u), len(traits_BM))

30000 30000 30000


In [ ]:
################################################################################################################################################
# We will start with traits simulated under the BM model.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and BM traits only).

# function to train on the combined datasets
def combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_BM_train)
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_BM_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_BM_test, xtest], ytest),callbacks=[earlyStopping])

    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')

    return model

# function to train on the SNP only datasets
def SNP_subset(ytrain, ytest, xtrain, xtest):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]

    # Create the the CNN 
    snps = create_cnn(xtest)
    
    #Create the last layer for the SNP network
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(xtrain, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(xtest, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    trait = create_cnn(traits_BM_train)
    
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_BM_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_BM_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

In [5]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [6]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d (Conv1D)                 (None, 98, 256)      23040       input_1[0][0]                    
__________________________________________________________________________________________________
conv1d_3 (Conv1D)               (None, 998, 256)     46080       input_2[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0111 - accuracy: 0.9971 - val_loss: 0.0011 - val_accuracy: 0.9997
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0106 - accuracy: 0.9974 - val_loss: 8.2368e-04 - val_accuracy: 0.9997
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0093 - accuracy: 0.9975 - val_loss: 8.1545e-04 - val_accuracy: 0.9997
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0066 - accuracy: 0.9983 - val_loss: 7.5558e-04 - val_accuracy: 0.9997
Epoch 18/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0055 - accuracy: 0.9987 - val_loss: 7.8090e-04 - val_accuracy: 0.9997
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0061 - accuracy: 0.9983 - val_loss: 7.1409e-04 - val_accuracy: 0.9997
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0058 - accuracy: 0.9983 - val_l

Epoch 70/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0012 - accuracy: 0.9996 - val_loss: 5.6184e-04 - val_accuracy: 0.9999
Epoch 71/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0012 - accuracy: 0.9996 - val_loss: 5.8664e-04 - val_accuracy: 0.9999
Epoch 72/100
88/88 [==============================] - 22s 249ms/step - loss: 7.0343e-04 - accuracy: 0.9998 - val_loss: 4.0037e-04 - val_accuracy: 0.9999
Final learned weights: Traits=0.5260, SNPs=0.4740
INFO:tensorflow:Assets written to: ./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100BM_50SNPs.mod/assets


In [7]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_20_Trained_CNN_Model_50SNPs.mod')

Model: "model_4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_6 (Conv1D)            (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_6 (Batch (None, 998, 256)          1024      
_________________________________________________________________
conv1d_7 (Conv1D)            (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_7 (Batch (None, 996, 256)          1024      
_________________________________________________________________
conv1d_8 (Conv1D)            (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_8 (Batch (None, 994, 256)          1024

In [8]:
################################################################################################################################################
# 40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [9]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_4 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_5 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_9 (Conv1D)               (None, 98, 256)      23040       input_4[0][0]                    
__________________________________________________________________________________________________
conv1d_12 (Conv1D)              (None, 998, 256)     46080       input_5[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0213 - accuracy: 0.9943 - val_loss: 0.0112 - val_accuracy: 0.9979
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0204 - accuracy: 0.9941 - val_loss: 0.0101 - val_accuracy: 0.9979
Epoch 16/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0163 - accuracy: 0.9957 - val_loss: 0.0104 - val_accuracy: 0.9980
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0128 - accuracy: 0.9964 - val_loss: 0.0099 - val_accuracy: 0.9981
Epoch 18/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0133 - accuracy: 0.9965 - val_loss: 0.0098 - val_accuracy: 0.9979
Epoch 19/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0135 - accuracy: 0.9960 - val_loss: 0.0090 - val_accuracy: 0.9981
Epoch 20/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0119 - accuracy: 0.9964 - val_loss: 0.0092 - val_ac

In [10]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest  = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_40_Trained_CNN_Model_50SNPs.mod')

Model: "model_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_6 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_15 (Conv1D)           (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_15 (Batc (None, 998, 256)          1024      
_________________________________________________________________
conv1d_16 (Conv1D)           (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_16 (Batc (None, 996, 256)          1024      
_________________________________________________________________
conv1d_17 (Conv1D)           (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_17 (Batc (None, 994, 256)          1024

88/88 [==============================] - 20s 228ms/step - loss: 0.0204 - accuracy: 0.9944 - val_loss: 0.0048 - val_accuracy: 0.9987
Epoch 100/100
88/88 [==============================] - 20s 227ms/step - loss: 0.0204 - accuracy: 0.9934 - val_loss: 0.0047 - val_accuracy: 0.9987
Time: 2011.0437080860138
INFO:tensorflow:Assets written to: ./Trained_Models/MD_Ind_SNPs_40_Trained_CNN_Model_50SNPs.mod/assets


In [11]:
################################################################################################################################################
# 60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [12]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_12"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_8 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_18 (Conv1D)              (None, 98, 256)      23040       input_7[0][0]                    
__________________________________________________________________________________________________
conv1d_21 (Conv1D)              (None, 998, 256)     46080       input_8[0][0]                    
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0195 - accuracy: 0.9951 - val_loss: 0.0115 - val_accuracy: 0.9979
Epoch 15/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0188 - accuracy: 0.9943 - val_loss: 0.0108 - val_accuracy: 0.9981
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0167 - accuracy: 0.9953 - val_loss: 0.0107 - val_accuracy: 0.9980
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0153 - accuracy: 0.9960 - val_loss: 0.0109 - val_accuracy: 0.9979
Epoch 18/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0120 - accuracy: 0.9968 - val_loss: 0.0106 - val_accuracy: 0.9980
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0138 - accuracy: 0.9964 - val_loss: 0.0103 - val_accuracy: 0.9983
Epoch 20/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0107 - accuracy: 0.9968 - val_loss: 0.0101 - val_ac

In [15]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_60_Trained_CNN_Model_50SNPs.mod')

Model: "model_14"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_9 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_24 (Conv1D)           (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_24 (Batc (None, 998, 256)          1024      
_________________________________________________________________
conv1d_25 (Conv1D)           (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_25 (Batc (None, 996, 256)          1024      
_________________________________________________________________
conv1d_26 (Conv1D)           (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_26 (Batc (None, 994, 256)          102

In [29]:
################################################################################################################################################
#20% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

In [30]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_20_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_32"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_19 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_20 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_54 (Conv1D)              (None, 98, 256)      23040       input_19[0][0]                   
__________________________________________________________________________________________________
conv1d_57 (Conv1D)              (None, 998, 256)     46080       input_20[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0086 - accuracy: 0.9978 - val_loss: 3.5738e-04 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 29s 331ms/step - loss: 0.0054 - accuracy: 0.9988 - val_loss: 2.9119e-04 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0066 - accuracy: 0.9982 - val_loss: 3.5599e-04 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 29s 328ms/step - loss: 0.0055 - accuracy: 0.9985 - val_loss: 2.5157e-04 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 29s 327ms/step - loss: 0.0050 - accuracy: 0.9989 - val_loss: 2.6678e-04 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0044 - accuracy: 0.9990 - val_loss: 2.4089e-04 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 29s 331ms/step - loss: 0.0052 - accuracy: 0.9985 - v

In [31]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_20_Trained_Traits_Model_100BM.mod')

Model: "model_34"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_21 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_60 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_60 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_61 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_61 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_62 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_62 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0053 - accuracy: 0.9989 - val_loss: 0.0187 - val_accuracy: 0.9955
Epoch 44/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0065 - accuracy: 0.9984 - val_loss: 0.0181 - val_accuracy: 0.9955
Epoch 45/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0052 - accuracy: 0.9988 - val_loss: 0.0178 - val_accuracy: 0.9955
Epoch 46/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0050 - accuracy: 0.9988 - val_loss: 0.0189 - val_accuracy: 0.9956
Epoch 47/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0054 - accuracy: 0.9988 - val_loss: 0.0192 - val_accuracy: 0.9956
Epoch 48/100
88/88 [==============================] - 2s 22ms/step - loss: 0.0051 - accuracy: 0.9986 - val_loss: 0.0192 - val_accuracy: 0.9959
Epoch 49/100
88/88 [==============================] - 2s 22ms/step - loss: 0.0053 - accuracy: 0.9988 - val_loss: 0.0201 - val_accuracy: 0.9957

In [32]:
################################################################################################################################################
#40% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

In [33]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_40_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_37"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_22 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_23 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_63 (Conv1D)              (None, 98, 256)      23040       input_22[0][0]                   
__________________________________________________________________________________________________
conv1d_66 (Conv1D)              (None, 998, 256)     46080       input_23[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0074 - accuracy: 0.9982 - val_loss: 2.9263e-04 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0061 - accuracy: 0.9988 - val_loss: 3.5356e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 29s 328ms/step - loss: 0.0059 - accuracy: 0.9986 - val_loss: 5.5886e-04 - val_accuracy: 0.9996
Epoch 17/100
88/88 [==============================] - 29s 328ms/step - loss: 0.0051 - accuracy: 0.9986 - val_loss: 2.6888e-04 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 29s 328ms/step - loss: 0.0052 - accuracy: 0.9986 - val_loss: 3.2057e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0043 - accuracy: 0.9989 - val_loss: 2.9694e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0046 - accuracy: 0.9988 - v

In [34]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_40_Trained_Traits_Model_100BM.mod')

Model: "model_39"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_24 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_69 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_69 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_70 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_70 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_71 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_71 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 22ms/step - loss: 0.0063 - accuracy: 0.9985 - val_loss: 0.0591 - val_accuracy: 0.9885
Epoch 44/100
88/88 [==============================] - 2s 22ms/step - loss: 0.0070 - accuracy: 0.9980 - val_loss: 0.0570 - val_accuracy: 0.9887
Epoch 45/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0051 - accuracy: 0.9988 - val_loss: 0.0584 - val_accuracy: 0.9892
Epoch 46/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0054 - accuracy: 0.9989 - val_loss: 0.0595 - val_accuracy: 0.9892
Epoch 47/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0054 - accuracy: 0.9982 - val_loss: 0.0599 - val_accuracy: 0.9892
Epoch 48/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0060 - accuracy: 0.9985 - val_loss: 0.0586 - val_accuracy: 0.9895
Epoch 49/100
88/88 [==============================] - 2s 23ms/step - loss: 0.0057 - accuracy: 0.9988 - val_loss: 0.0593 - val_accuracy: 0.9891

In [35]:
################################################################################################################################################
# 60% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

In [36]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_60_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_42"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_25 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_26 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_72 (Conv1D)              (None, 98, 256)      23040       input_25[0][0]                   
__________________________________________________________________________________________________
conv1d_75 (Conv1D)              (None, 998, 256)     46080       input_26[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0085 - accuracy: 0.9978 - val_loss: 3.4639e-04 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 29s 331ms/step - loss: 0.0073 - accuracy: 0.9984 - val_loss: 3.5350e-04 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0067 - accuracy: 0.9982 - val_loss: 2.8672e-04 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 29s 327ms/step - loss: 0.0058 - accuracy: 0.9986 - val_loss: 3.1996e-04 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0059 - accuracy: 0.9986 - val_loss: 2.8009e-04 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0049 - accuracy: 0.9990 - val_loss: 1.7793e-04 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0041 - accuracy: 0.9990 - v

In [37]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_BM_60_Trained_Traits_Model_100BM.mod')

Model: "model_44"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_27 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_78 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_78 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_79 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_79 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_80 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_80 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 20ms/step - loss: 0.0115 - accuracy: 0.9969 - val_loss: 0.0839 - val_accuracy: 0.9808
Epoch 44/100
88/88 [==============================] - 2s 20ms/step - loss: 0.0106 - accuracy: 0.9972 - val_loss: 0.0863 - val_accuracy: 0.9812
Epoch 45/100
88/88 [==============================] - 2s 20ms/step - loss: 0.0097 - accuracy: 0.9974 - val_loss: 0.0856 - val_accuracy: 0.9812
Epoch 46/100
88/88 [==============================] - 2s 20ms/step - loss: 0.0098 - accuracy: 0.9976 - val_loss: 0.0853 - val_accuracy: 0.9816
Epoch 47/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0091 - accuracy: 0.9976 - val_loss: 0.0887 - val_accuracy: 0.9812
Epoch 48/100
88/88 [==============================] - 2s 20ms/step - loss: 0.0102 - accuracy: 0.9969 - val_loss: 0.0882 - val_accuracy: 0.9819
Epoch 49/100
88/88 [==============================] - 2s 21ms/step - loss: 0.0091 - accuracy: 0.9978 - val_loss: 0.0905 - val_accuracy: 0.9812

In [ ]:
################################################################################################################################################
# Now repeat with traits simulated under the OU model.
################################################################################################################################################
# load the traits simulated under the OU model for the 3 scenarios. 
traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
Traits_OU = np.array(traits_OU)

# standard scale the continuous (OU) traits
scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i]) 

# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./trainingSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./trainingSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./trainingSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
X=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

#transform major alleles in -1 and minor in 1
for arr,array in enumerate(X):
    for idx,row in enumerate(array):
        if np.count_nonzero(row) > len(row)/2:
            X[arr][idx][X[arr][idx] == 1] = -1
            X[arr][idx][X[arr][idx] == 0] = 1
        else:
            X[arr][idx][X[arr][idx] == 0] = -1
            
# create a label vector in the same order as the simulations.
y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2sp']))])
y.extend([2 for i in range(len(u3['Model_3sp']))])
y = np.array(y)

# make sure labels, SNP and traits matrices all have the same length.
print (len(X), len(y), len(traits_OU))                                                                                                        # Create the MLP and CNN model

In [ ]:
# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and OU traits only).

# function to train on the combined datasets
def combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_OU_train)
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_OU_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_OU_test, xtest], ytest),callbacks=[earlyStopping])
    
    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the OU trait only datasets
def OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    trait = create_cnn(traits_OU_train)

    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_OU_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_OU_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

In [40]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [41]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_47"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_28 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_29 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_81 (Conv1D)              (None, 98, 256)      23040       input_28[0][0]                   
__________________________________________________________________________________________________
conv1d_84 (Conv1D)              (None, 998, 256)     46080       input_29[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0090 - accuracy: 0.9976 - val_loss: 3.9277e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0076 - accuracy: 0.9980 - val_loss: 3.8407e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0067 - accuracy: 0.9983 - val_loss: 3.5271e-04 - val_accuracy: 0.9999
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0060 - accuracy: 0.9987 - val_loss: 3.8092e-04 - val_accuracy: 0.9999
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0063 - accuracy: 0.9985 - val_loss: 3.6166e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 2.5444e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0047 - accuracy: 0.9988 - v

In [42]:
################################################################################################################################################
#40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [43]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_50"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_30 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_31 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_87 (Conv1D)              (None, 98, 256)      23040       input_30[0][0]                   
__________________________________________________________________________________________________
conv1d_90 (Conv1D)              (None, 998, 256)     46080       input_31[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0172 - accuracy: 0.9958 - val_loss: 0.0120 - val_accuracy: 0.9977
Epoch 15/100
88/88 [==============================] - 29s 327ms/step - loss: 0.0163 - accuracy: 0.9962 - val_loss: 0.0126 - val_accuracy: 0.9980
Epoch 16/100
88/88 [==============================] - 29s 332ms/step - loss: 0.0146 - accuracy: 0.9959 - val_loss: 0.0114 - val_accuracy: 0.9981
Epoch 17/100
88/88 [==============================] - 29s 327ms/step - loss: 0.0128 - accuracy: 0.9967 - val_loss: 0.0110 - val_accuracy: 0.9983
Epoch 18/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0120 - accuracy: 0.9964 - val_loss: 0.0103 - val_accuracy: 0.9981
Epoch 19/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0129 - accuracy: 0.9964 - val_loss: 0.0101 - val_accuracy: 0.9981
Epoch 20/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0102 - accuracy: 0.9975 - val_loss: 0.0108 - val_ac

In [44]:
################################################################################################################################################
#60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [45]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_53"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_32 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_33 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_93 (Conv1D)              (None, 98, 256)      23040       input_32[0][0]                   
__________________________________________________________________________________________________
conv1d_96 (Conv1D)              (None, 998, 256)     46080       input_33[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0191 - accuracy: 0.9948 - val_loss: 0.0062 - val_accuracy: 0.9975
Epoch 15/100
88/88 [==============================] - 29s 329ms/step - loss: 0.0159 - accuracy: 0.9958 - val_loss: 0.0059 - val_accuracy: 0.9981
Epoch 16/100
88/88 [==============================] - 29s 331ms/step - loss: 0.0154 - accuracy: 0.9955 - val_loss: 0.0064 - val_accuracy: 0.9977
Epoch 17/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0123 - accuracy: 0.9964 - val_loss: 0.0058 - val_accuracy: 0.9980
Epoch 18/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0118 - accuracy: 0.9966 - val_loss: 0.0053 - val_accuracy: 0.9984
Epoch 19/100
88/88 [==============================] - 29s 331ms/step - loss: 0.0104 - accuracy: 0.9968 - val_loss: 0.0051 - val_accuracy: 0.9983
Epoch 20/100
88/88 [==============================] - 29s 330ms/step - loss: 0.0096 - accuracy: 0.9975 - val_loss: 0.0045 - val_ac

In [71]:
################################################################################################################################################
# 20% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

In [47]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_20_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_56"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_34 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_35 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_99 (Conv1D)              (None, 98, 256)      23040       input_34[0][0]                   
__________________________________________________________________________________________________
conv1d_102 (Conv1D)             (None, 998, 256)     46080       input_35[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 27s 302ms/step - loss: 0.0492 - accuracy: 0.9860 - val_loss: 0.0266 - val_accuracy: 0.9927
Epoch 15/100
88/88 [==============================] - 27s 302ms/step - loss: 0.0361 - accuracy: 0.9893 - val_loss: 0.0231 - val_accuracy: 0.9932
Epoch 16/100
88/88 [==============================] - 26s 302ms/step - loss: 0.0340 - accuracy: 0.9905 - val_loss: 0.0213 - val_accuracy: 0.9949
Epoch 17/100
88/88 [==============================] - 27s 303ms/step - loss: 0.0296 - accuracy: 0.9909 - val_loss: 0.0198 - val_accuracy: 0.9947
Epoch 18/100
88/88 [==============================] - 27s 302ms/step - loss: 0.0260 - accuracy: 0.9925 - val_loss: 0.0194 - val_accuracy: 0.9952
Epoch 19/100
88/88 [==============================] - 27s 302ms/step - loss: 0.0239 - accuracy: 0.9935 - val_loss: 0.0193 - val_accuracy: 0.9955
Epoch 20/100
88/88 [==============================] - 27s 303ms/step - loss: 0.0193 - accuracy: 0.9942 - val_loss: 0.0186 - val_ac

In [72]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_20_Trained_Traits_Model_100OU.mod')

Model: "model_82"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_51 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_150 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_150 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_151 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_151 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_152 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_152 (Bat (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0040 - accuracy: 0.9992 - val_loss: 0.0165 - val_accuracy: 0.9964
Epoch 44/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0037 - accuracy: 0.9992 - val_loss: 0.0160 - val_accuracy: 0.9965
Epoch 45/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0036 - accuracy: 0.9990 - val_loss: 0.0158 - val_accuracy: 0.9967
Epoch 46/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0044 - accuracy: 0.9989 - val_loss: 0.0157 - val_accuracy: 0.9968
Epoch 47/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0032 - accuracy: 0.9992 - val_loss: 0.0159 - val_accuracy: 0.9967
Epoch 48/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0034 - accuracy: 0.9992 - val_loss: 0.0166 - val_accuracy: 0.9964
Epoch 49/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0029 - accuracy: 0.9994 - val_loss: 0.0166 - val_accuracy: 0.9964

In [73]:
################################################################################################################################################
# 40% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

In [74]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_40_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_85"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_52 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_53 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_153 (Conv1D)             (None, 98, 256)      23040       input_52[0][0]                   
__________________________________________________________________________________________________
conv1d_156 (Conv1D)             (None, 998, 256)     46080       input_53[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0075 - accuracy: 0.9981 - val_loss: 5.1451e-05 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0067 - accuracy: 0.9981 - val_loss: 7.3839e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0065 - accuracy: 0.9980 - val_loss: 4.7114e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0045 - accuracy: 0.9993 - val_loss: 4.6434e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0053 - accuracy: 0.9985 - val_loss: 5.8194e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0044 - accuracy: 0.9990 - val_loss: 2.9452e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0041 - accuracy: 0.9989 - v

In [75]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_40_Trained_Traits_Model_100OU.mod')

Model: "model_87"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_54 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_159 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_159 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_160 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_160 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_161 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_161 (Bat (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0056 - accuracy: 0.9988 - val_loss: 0.0395 - val_accuracy: 0.9907
Epoch 44/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0054 - accuracy: 0.9986 - val_loss: 0.0381 - val_accuracy: 0.9909
Epoch 45/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0051 - accuracy: 0.9986 - val_loss: 0.0391 - val_accuracy: 0.9907
Epoch 46/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0050 - accuracy: 0.9988 - val_loss: 0.0380 - val_accuracy: 0.9915
Epoch 47/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0059 - accuracy: 0.9983 - val_loss: 0.0373 - val_accuracy: 0.9912
Epoch 48/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0056 - accuracy: 0.9986 - val_loss: 0.0379 - val_accuracy: 0.9912
Epoch 49/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0061 - accuracy: 0.9982 - val_loss: 0.0390 - val_accuracy: 0.9909

In [102]:
################################################################################################################################################
# 60% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

In [105]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_60_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_122"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_75 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_76 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_222 (Conv1D)             (None, 98, 256)      23040       input_75[0][0]                   
__________________________________________________________________________________________________
conv1d_225 (Conv1D)             (None, 998, 256)     46080       input_76[0][0]                   
__________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 26s 295ms/step - loss: 0.0074 - accuracy: 0.9981 - val_loss: 1.2031e-04 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 26s 296ms/step - loss: 0.0064 - accuracy: 0.9986 - val_loss: 9.7553e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 27s 303ms/step - loss: 0.0064 - accuracy: 0.9984 - val_loss: 8.7957e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 26s 295ms/step - loss: 0.0050 - accuracy: 0.9984 - val_loss: 7.1091e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 26s 295ms/step - loss: 0.0058 - accuracy: 0.9985 - val_loss: 5.0257e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 26s 301ms/step - loss: 0.0049 - accuracy: 0.9987 - val_loss: 7.0003e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 26s 297ms/step - loss: 0.0048 - accuracy: 0.9990 - v

In [78]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_OU_60_Trained_Traits_Model_100OU.mod')

88/88 [==============================] - 2s 17ms/step - loss: 0.0043 - accuracy: 0.9989 - val_loss: 0.0796 - val_accuracy: 0.9847
Epoch 67/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0041 - accuracy: 0.9992 - val_loss: 0.0806 - val_accuracy: 0.9844
Epoch 68/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0047 - accuracy: 0.9988 - val_loss: 0.0800 - val_accuracy: 0.9851
Epoch 69/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0036 - accuracy: 0.9992 - val_loss: 0.0805 - val_accuracy: 0.9847
Epoch 70/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 0.0804 - val_accuracy: 0.9839
Epoch 71/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0048 - accuracy: 0.9990 - val_loss: 0.0808 - val_accuracy: 0.9836
Epoch 72/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0044 - accuracy: 0.9988 - val_loss: 0.0799 - val_accuracy: 0.9843
Epoch 73/100

In [ ]:
################################################################################################################################################
# Now repeat with discrete traits.
################################################################################################################################################

# load the discrete traits simulated for the 3 scenarios. 
traits_disc = []
traits_disc = np.loadtxt("./traits/traits_disc.txt").reshape(30000,-1,100)
# transform into a NumPy array.
Traits_disc = np.array(traits_disc)

# load the SNPs simulated for the 3 scenarios.
u1 = np.load("./trainingSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./trainingSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./trainingSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
X=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

#transform major alleles in -1 and minor in 1
for arr,array in enumerate(X):
    for idx,row in enumerate(array):
        if np.count_nonzero(row) > len(row)/2:
            X[arr][idx][X[arr][idx] == 1] = -1
            X[arr][idx][X[arr][idx] == 0] = 1
        else:
            X[arr][idx][X[arr][idx] == 0] = -1
            
# create a label vector in the same order as the simulations.
y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2sp']))])
y.extend([2 for i in range(len(u3['Model_3sp']))])
y = np.array(y)

# make sure labels, SNP and traits matrices all have the same length.
print (len(X), len(y), len(traits_disc))

In [ ]:
# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and discrete traits only).

# function to train on the combined datasets
def combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_disc_train=np.swapaxes(traits_disc_train, 1, 2)
    traits_disc_test=np.swapaxes(traits_disc_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_disc_train)
    snps = create_cnn(xtrain)
    
    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_disc_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_disc_test, xtest], ytest),callbacks=[earlyStopping])

   # Get contributions from each branch.
    gated_contributions(model)    
    print (f'Time: {time.time() - start}')
        
    return model

# function to train on the discrete trait only datasets
def disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_disc_train=np.swapaxes(traits_disc_train, 1, 2)
    traits_disc_test=np.swapaxes(traits_disc_test, 1, 2)
    trait = create_cnn(traits_disc_train)
    
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_disc_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_disc_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

In [81]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [82]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_95"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_58 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_59 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_171 (Conv1D)             (None, 98, 256)      23040       input_58[0][0]                   
__________________________________________________________________________________________________
conv1d_174 (Conv1D)             (None, 998, 256)     46080       input_59[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0073 - accuracy: 0.9982 - val_loss: 1.5780e-04 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0065 - accuracy: 0.9987 - val_loss: 1.5107e-04 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0065 - accuracy: 0.9982 - val_loss: 1.2128e-04 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0053 - accuracy: 0.9986 - val_loss: 8.1327e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0044 - accuracy: 0.9990 - val_loss: 6.7042e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0045 - accuracy: 0.9987 - val_loss: 5.5874e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 251ms/step - loss: 0.0040 - accuracy: 0.9990 - v

In [83]:
################################################################################################################################################
#40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [84]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_98"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_60 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_61 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_177 (Conv1D)             (None, 98, 256)      23040       input_60[0][0]                   
__________________________________________________________________________________________________
conv1d_180 (Conv1D)             (None, 998, 256)     46080       input_61[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0161 - accuracy: 0.9955 - val_loss: 0.0049 - val_accuracy: 0.9984
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0142 - accuracy: 0.9962 - val_loss: 0.0042 - val_accuracy: 0.9987
Epoch 16/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0148 - accuracy: 0.9956 - val_loss: 0.0038 - val_accuracy: 0.9987
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0113 - accuracy: 0.9971 - val_loss: 0.0037 - val_accuracy: 0.9987
Epoch 18/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0104 - accuracy: 0.9971 - val_loss: 0.0033 - val_accuracy: 0.9989
Epoch 19/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0091 - accuracy: 0.9972 - val_loss: 0.0031 - val_accuracy: 0.9991
Epoch 20/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0084 - accuracy: 0.9975 - val_loss: 0.0033 - val_ac

In [7]:
################################################################################################################################################
#60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
  for m in range(missD):
    k = random.randint(0, X.shape[2] - 1)
    X[i,:,k] = 0

In [ ]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 100, 30)]            0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 1000, 60)]           0         []                            
                                                                                                  
 conv1d (Conv1D)             (None, 98, 256)              23040     ['input_1[0][0]']             
                                                                                                  
 conv1d_3 (Conv1D)           (None, 998, 256)             46080     ['input_2[0][0]']             
                                                                                            

In [87]:
################################################################################################################################################
# 20% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

In [88]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_20_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_104"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_64 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_65 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_189 (Conv1D)             (None, 98, 256)      23040       input_64[0][0]                   
__________________________________________________________________________________________________
conv1d_192 (Conv1D)             (None, 998, 256)     46080       input_65[0][0]                   
__________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.8197 - accuracy: 0.6140 - val_loss: 0.6432 - val_accuracy: 0.6849
Epoch 15/100
88/88 [==============================] - 22s 250ms/step - loss: 0.6315 - accuracy: 0.6990 - val_loss: 0.5993 - val_accuracy: 0.6704
Epoch 16/100
88/88 [==============================] - 22s 250ms/step - loss: 0.5472 - accuracy: 0.7358 - val_loss: 0.5464 - val_accuracy: 0.6907
Epoch 17/100
88/88 [==============================] - 22s 250ms/step - loss: 0.5056 - accuracy: 0.7543 - val_loss: 0.5054 - val_accuracy: 0.7051
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4749 - accuracy: 0.7669 - val_loss: 0.4425 - val_accuracy: 0.7732
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4482 - accuracy: 0.7817 - val_loss: 0.4357 - val_accuracy: 0.7645
Epoch 20/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4335 - accuracy: 0.7914 - val_loss: 0.3862 - val_ac

In [89]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_20_Trained_Traits_Model_100discrete.mod')

Model: "model_106"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_66 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_195 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_195 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_196 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_196 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_197 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_197 (Bat (None, 94, 256)           10

Epoch 43/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0131 - accuracy: 0.9967 - val_loss: 0.0232 - val_accuracy: 0.9929
Epoch 44/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0150 - accuracy: 0.9956 - val_loss: 0.0220 - val_accuracy: 0.9936
Epoch 45/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0150 - accuracy: 0.9959 - val_loss: 0.0228 - val_accuracy: 0.9935
Epoch 46/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0121 - accuracy: 0.9968 - val_loss: 0.0221 - val_accuracy: 0.9929
Epoch 47/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0132 - accuracy: 0.9964 - val_loss: 0.0218 - val_accuracy: 0.9931
Epoch 48/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0118 - accuracy: 0.9970 - val_loss: 0.0223 - val_accuracy: 0.9933
Epoch 49/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0125 - accuracy: 0.9968 - val_loss: 0.0229 - val_accuracy: 0.9925

In [90]:
################################################################################################################################################
# 40% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

In [91]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_40_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_109"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_67 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_68 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_198 (Conv1D)             (None, 98, 256)      23040       input_67[0][0]                   
__________________________________________________________________________________________________
conv1d_201 (Conv1D)             (None, 998, 256)     46080       input_68[0][0]                   
__________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4824 - accuracy: 0.7809 - val_loss: 0.4840 - val_accuracy: 0.7664
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4406 - accuracy: 0.8002 - val_loss: 0.4325 - val_accuracy: 0.8143
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4156 - accuracy: 0.8141 - val_loss: 0.3923 - val_accuracy: 0.8455
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.3840 - accuracy: 0.8333 - val_loss: 0.3672 - val_accuracy: 0.8653
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.3468 - accuracy: 0.8569 - val_loss: 0.2794 - val_accuracy: 0.9091
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.3006 - accuracy: 0.8821 - val_loss: 0.1961 - val_accuracy: 0.9407
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.2426 - accuracy: 0.9108 - val_loss: 0.1396 - val_ac

In [92]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_40_Trained_Traits_Model_100discrete.mod')

Model: "model_111"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_69 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_204 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_204 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_205 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_205 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_206 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_206 (Bat (None, 94, 256)           10

Epoch 43/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0282 - accuracy: 0.9912 - val_loss: 0.0449 - val_accuracy: 0.9867
Epoch 44/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0242 - accuracy: 0.9931 - val_loss: 0.0442 - val_accuracy: 0.9872
Epoch 45/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0267 - accuracy: 0.9914 - val_loss: 0.0487 - val_accuracy: 0.9855
Epoch 46/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0243 - accuracy: 0.9924 - val_loss: 0.0486 - val_accuracy: 0.9857
Epoch 47/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0243 - accuracy: 0.9929 - val_loss: 0.0421 - val_accuracy: 0.9875
Epoch 48/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0222 - accuracy: 0.9942 - val_loss: 0.0433 - val_accuracy: 0.9851
Epoch 49/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0241 - accuracy: 0.9932 - val_loss: 0.0447 - val_accuracy: 0.9857

In [93]:
################################################################################################################################################
# 60% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=u

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

In [94]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_60_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_114"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_70 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_71 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_207 (Conv1D)             (None, 98, 256)      23040       input_70[0][0]                   
__________________________________________________________________________________________________
conv1d_210 (Conv1D)             (None, 998, 256)     46080       input_71[0][0]                   
__________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4817 - accuracy: 0.7767 - val_loss: 0.4801 - val_accuracy: 0.8112
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4561 - accuracy: 0.7888 - val_loss: 0.4333 - val_accuracy: 0.8197
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4399 - accuracy: 0.7955 - val_loss: 0.4394 - val_accuracy: 0.8161
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4237 - accuracy: 0.8029 - val_loss: 0.4255 - val_accuracy: 0.8240
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.4061 - accuracy: 0.8121 - val_loss: 0.3954 - val_accuracy: 0.8411
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.3936 - accuracy: 0.8203 - val_loss: 0.3890 - val_accuracy: 0.8524
Epoch 20/100
88/88 [==============================] - 22s 249ms/step - loss: 0.3822 - accuracy: 0.8267 - val_loss: 0.3631 - val_ac

In [95]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_Ind_discrete_60_Trained_Traits_Model_100discrete.mod')

Model: "model_116"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_72 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_213 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_213 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_214 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_214 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_215 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_215 (Bat (None, 94, 256)           10

Epoch 43/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0856 - accuracy: 0.9705 - val_loss: 0.1022 - val_accuracy: 0.9663
Epoch 44/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0777 - accuracy: 0.9731 - val_loss: 0.1039 - val_accuracy: 0.9665
Epoch 45/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0755 - accuracy: 0.9747 - val_loss: 0.1040 - val_accuracy: 0.9667
Epoch 46/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0710 - accuracy: 0.9754 - val_loss: 0.1016 - val_accuracy: 0.9681
Epoch 47/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0648 - accuracy: 0.9784 - val_loss: 0.1010 - val_accuracy: 0.9681
Epoch 48/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0582 - accuracy: 0.9800 - val_loss: 0.1238 - val_accuracy: 0.9647
Epoch 49/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0534 - accuracy: 0.9818 - val_loss: 0.1071 - val_accuracy: 0.9672

In [11]:
# Now that the models are trained, we will evaluate their accuracy based on the test set. For that, we will build confusion matrices containing the true and predicted scenarions for each simulation on the test set.

# first import the libraries
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from keras.models import load_model
from sklearn.metrics import confusion_matrix

# load the trained models.
model1 = load_model('./Trained_Models/MD_Ind_BM_20_Trained_Traits_Model_100BM.acc.mod')
model2 = load_model('./Trained_Models/MD_Ind_OU_20_Trained_Traits_Model_100OU.acc.mod')
model3 = load_model('./Trained_Models/MD_Ind_discrete_20_Trained_Traits_Model_100discrete.acc.mod')
model4 = load_model('./Trained_Models/MD_Ind_SNPs_20_Trained_CNN_Model_50SNPs.acc.mod')
model5 = load_model('./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model6 = load_model('./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model7 = load_model('./Trained_Models/MD_Ind_SNPs_20_Trained_Comb_Model_100discrete_50SNPs.acc.mod')
model8 = load_model('./Trained_Models/MD_Ind_BM_20_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model9 = load_model('./Trained_Models/MD_Ind_OU_20_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model10 = load_model('./Trained_Models/MD_Ind_discrete_20_Trained_Comb_Model_100discrete_50SNPs.acc.mod')
model11 = load_model('./Trained_Models/MD_Ind_BM_40_Trained_Traits_Model_100BM.acc.mod')
model12 = load_model('./Trained_Models/MD_Ind_OU_40_Trained_Traits_Model_100OU.acc.mod')
model13 = load_model('./Trained_Models/MD_Ind_discrete_40_Trained_Traits_Model_100discrete.acc.mod')
model14 = load_model('./Trained_Models/MD_Ind_SNPs_40_Trained_CNN_Model_50SNPs.acc.mod')
model15 = load_model('./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model16 = load_model('./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model17 = load_model('./Trained_Models/MD_Ind_SNPs_40_Trained_Comb_Model_100discrete_50SNPs.acc.mod')
model18 = load_model('./Trained_Models/MD_Ind_BM_40_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model19 = load_model('./Trained_Models/MD_Ind_OU_40_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model20 = load_model('./Trained_Models/MD_Ind_discrete_40_Trained_Comb_Model_100discrete_50SNPs.acc.mod')
model21 = load_model('./Trained_Models/MD_Ind_BM_60_Trained_Traits_Model_100BM.acc.mod')
model22 = load_model('./Trained_Models/MD_Ind_OU_60_Trained_Traits_Model_100OU.acc.mod')
model23 = load_model('./Trained_Models/MD_Ind_discrete_60_Trained_Traits_Model_100discrete.acc.mod')
model24 = load_model('./Trained_Models/MD_Ind_SNPs_60_Trained_CNN_Model_50SNPs.acc.mod')
model25 = load_model('./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model26 = load_model('./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model27 = load_model('./Trained_Models/MD_Ind_SNPs_60_Trained_Comb_Model_100discrete_50SNPs.acc.mod')
model28 = load_model('./Trained_Models/MD_Ind_BM_60_Trained_Comb_Model_100BM_50SNPs.acc.mod')
model29 = load_model('./Trained_Models/MD_Ind_OU_60_Trained_Comb_Model_100OU_50SNPs.acc.mod')
model30 = load_model('./Trained_Models/MD_Ind_discrete_60_Trained_Comb_Model_100discrete_50SNPs.acc.mod')

In [13]:
# load the traits simulated under the BM model for the 3 scenarios.
traits_BM = []
traits_BM = np.loadtxt("./testSims/traits/traits_BM.txt").reshape(3000,-1,100)
# transform into a NumPy array. 
traits_BM = np.array(traits_BM)

#Standard scale the continuous variables
for i in range(traits_BM.shape[2]):
    traits_BM[:, :, i] = scalers_BM[i].transform(traits_BM[:, :, i]) 

# transform into a NumPy array. 
Traits_BM = np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

# transform into a NumPy array.
traits_BM20=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

# transform into a NumPy array.
traits_BM40=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_BM.shape[1]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_BM.shape[1] - 1)
    traits_BM[i,j,:] = 0

# transform into a NumPy array.
traits_BM60=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

# load the traits simulated under the OU model for the 3 scenarios.
traits_OU = []
traits_OU = np.loadtxt("./testSims/traits/traits_OU.txt").reshape(3000,-1,100)
# transform into a NumPy array. 
traits_OU = np.array(traits_OU)

#Standard scale the continuous variables
for i in range(traits_OU.shape[2]):
    traits_OU[:, :, i] = scalers_OU[i].transform(traits_OU[:, :, i]) 

# transform into a NumPy array. 
Traits_OU = np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

# transform into a NumPy array. 
traits_OU20=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

# transform into a NumPy array. 
traits_OU40=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_OU.shape[1]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_OU.shape[1] - 1)
    traits_OU[i,j,:] = 0

# transform into a NumPy array. 
traits_OU60=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

# load the discrete traits simulated for the 3 scenarios.
traits_disc = []
traits_disc = np.loadtxt("./testSims/traits/traits_disc.txt").reshape(3000,-1,100)
# transform into a NumPy array. 
traits_disc = np.array(traits_disc)
Traits_disc = np.array(traits_disc)
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

# transform into a NumPy array. 
traits_disc20=np.array(traits_disc)
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

# transform into a NumPy array. 
traits_disc40=np.array(traits_disc)
traits_disc=np.array(traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_disc.shape[1]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    traits_disc[i,j,:] = 0

# transform into a NumPy array. 
traits_disc60=np.array(traits_disc)
traits_disc=np.array(traits_disc)

# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./testSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./testSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./testSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
Xtest=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

#transform major alleles in -1 and minor in 1
for arr,array in enumerate(Xtest):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      Xtest[arr][idx][Xtest[arr][idx] == 1] = -1
      Xtest[arr][idx][Xtest[arr][idx] == 0] = 1
    else:
      Xtest[arr][idx][Xtest[arr][idx] == 0] = -1

# transform into a NumPy array. 
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i,:,k] = 0

# transform into a NumPy array.
xtest20=np.array(xtest)
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i,:,k] = 0

# transform into a NumPy array.
xtest40=np.array(xtest)
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i,:,k] = 0

# transform into a NumPy array.
xtest60=np.array(xtest)
xtest=np.array(Xtest)

# create a label vector in the same order as the simulations.  
ytest=[0 for i in range(len(u1['Model_1sp']))]
ytest.extend([1 for i in range(len(u2['Model_2sp']))])
ytest.extend([2 for i in range(len(u3['Model_3sp']))])
ytest = np.array(ytest)

In [14]:
#define a funtion to build the confusion matrix
def makeConfusionMatrixHeatmap(data, title, trueClassOrderLs, predictedClassOrderLs, ax):
    data = np.array(data)
    data = normalize(data, axis=1, norm='l1')
    heatmap = ax.pcolor(data, cmap=plt.cm.Blues, vmin=0.0, vmax=1.0)

    for i in range(len(predictedClassOrderLs)):
        for j in reversed(range(len(trueClassOrderLs))):
            val = 100*data[j, i]
            if val > 50:
                c = '0.9'
            else:
                c = 'black'
            ax.text(i + 0.5, j + 0.5, '%.2f%%' % val, horizontalalignment='center', verticalalignment='center', color=c, fontsize=9)

    cbar = plt.colorbar(heatmap, cmap=plt.cm.Blues, ax=ax)
    cbar.set_label("Fraction of simulations assigned to class", rotation=270, labelpad=20, fontsize=11)

    # put the major ticks at the middle of each cell
    ax.set_xticks(np.arange(data.shape[1]) + 0.5, minor=False)
    ax.set_yticks(np.arange(data.shape[0]) + 0.5, minor=False)
    ax.axis('tight')
    ax.set_title(title)

    #labels
    ax.set_xticklabels(predictedClassOrderLs, minor=False, fontsize=9, rotation=45)
    ax.set_yticklabels(reversed(trueClassOrderLs), minor=False, fontsize=9)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")

In [ ]:
# Now we will plot the confusion matrices for each trained model
#first get the predictions
pred_all = np.array([])
pred = model1.predict(np.swapaxes(traits_BM20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM20_pred_all.csv', pred, delimiter=',')
counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions for the next dataset
pred = model2.predict(np.swapaxes(traits_OU20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model3.predict(np.swapaxes(traits_disc20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model4.predict(xtest20)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model5.predict([np.swapaxes(traits_BM, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model6.predict([np.swapaxes(traits_OU, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model7.predict([np.swapaxes(traits_disc, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model8.predict([np.swapaxes(traits_BM20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model9.predict([np.swapaxes(traits_OU20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model10.predict([np.swapaxes(traits_disc20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model11.predict(np.swapaxes(traits_BM40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM40_pred_all.csv', pred, delimiter=',')
print (confusion_matrix(ytest, pred_cat))

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model12.predict(np.swapaxes(traits_OU40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU40_pred_all.csv', pred, delimiter=',')


counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model13.predict(np.swapaxes(traits_disc40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model14.predict(xtest40)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model15.predict([np.swapaxes(traits_BM, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

#now the actual work #6
#first get the predictions
pred = model16.predict([np.swapaxes(traits_OU, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model17.predict([np.swapaxes(traits_disc, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model18.predict([np.swapaxes(traits_BM40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model19.predict([np.swapaxes(traits_OU40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model20.predict([np.swapaxes(traits_disc40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model21.predict(np.swapaxes(traits_BM60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM60_pred_all.csv', pred, delimiter=',')
print (confusion_matrix(ytest, pred_cat))

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model22.predict(np.swapaxes(traits_OU60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU60_pred_all.csv', pred, delimiter=',')


counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model23.predict(np.swapaxes(traits_disc60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model24.predict(xtest60)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model25.predict([np.swapaxes(traits_BM, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model26.predict([np.swapaxes(traits_OU, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model27.predict([np.swapaxes(traits_disc, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model28.predict([np.swapaxes(traits_BM60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/BM60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model29.predict([np.swapaxes(traits_OU60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/OU60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model30.predict([np.swapaxes(traits_disc60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingSamples/disc60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()